# Week 59, Day 6: Forge: Week 59 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w059/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", round(np.sqrt(0.7 * 0.3 / 100), 4))
print("2.", round(1 / 201, 4))
print("3.", round(2 * 1 / (1 + 1), 2), "units above the mean")

### Exercise: Part A: nested, grouped and permutation-tested

Complete evaluate(X, y, sess): for each held-out session, choose the penalty by leave-one-session-out cross-validation among the three training sessions, refit on all three, and score on the held-out session. Then build the permutation null with labels shuffled within each session.

In [ ]:
import numpy as np
gA = np.random.default_rng(596)
w_true = gA.normal(size=30) / np.sqrt(30)
Xs, ys, ss = [], [], []
for s in range(4):
    yy = gA.integers(0, 2, 100)
    drift = np.cumsum(0.15 * gA.normal(size=(100, 30)), axis=0)
    Xs.append(1.2 * np.outer(2 * yy - 1, w_true) + 1.5 * gA.normal(size=30) + drift + gA.normal(size=(100, 30)))
    ys.append(yy); ss.append(np.full(100, s))
X, y, sess = np.vstack(Xs), np.concatenate(ys), np.concatenate(ss)
lams = np.logspace(0, 4, 6)

def fit_score(Xtr, ytr, Xte, yte, lam):
    mu, sd = Xtr.mean(0), Xtr.std(0)                    # scaling fitted on training trials only
    A, B = (Xtr - mu) / sd, (Xte - mu) / sd
    t = 2.0 * ytr - 1
    w = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ (t - t.mean()))
    return np.mean(((B @ w + t.mean()) > 0) == (yte == 1))

def evaluate(X, y, sess):
    outer = []
    for s_out in np.unique(sess):
        tr, te = sess != s_out, sess == s_out
        inner = []
        for lam in lams:
            inner.append(np.mean([fit_score(X[tr & (sess != s_in)], y[tr & (sess != s_in)],
                                            X[sess == s_in], y[sess == s_in], lam)
                                  for s_in in np.unique(sess[tr])]))
        lam_best = ...
        outer.append(...)
    return np.mean(outer)

nested_score = evaluate(X, y, sess)
gP = np.random.default_rng(5961)
null_scores = []
for k in range(40):
    y_perm = y.copy()
    for s in range(4):
        idx = np.where(sess == s)[0]
        y_perm[idx] = gP.permutation(y[idx])
    null_scores.append(evaluate(X, y_perm, sess))
p_value = (1 + np.sum(np.array(null_scores) >= nested_score)) / (1 + len(null_scores))
print(f"nested leave-one-session-out accuracy {nested_score:.3f}")
print(f"null mean {np.mean(null_scores):.3f}, null max {np.max(null_scores):.3f}, p = {p_value:.3f} (smallest possible {1 / 41:.3f})")

In [ ]:
import numpy as np
def test_eval():
    "the nested leave-one-session-out accuracy is above chance"
    assert 0.55 < nested_score < 0.9
def test_perm():
    "the permutation null is centred near chance and the p-value is below 0.05"
    assert len(null_scores) == 40 and abs(np.mean(null_scores) - 0.5) < 0.05 and p_value < 0.05

test_eval()
test_perm()
print("All checks passed")

### Exercise: Part B: the spread of cross-validation

For 200 datasets of 100 trials, compute the 5-fold nearest-centroid accuracy and store it. Compare the standard deviation of these estimates with sqrt(a(1 - a)/100), where a is their mean.

In [ ]:
def centroid_cv(Xd, yd, g):
    folds = np.array_split(g.permutation(len(yd)), 5)
    right = 0
    for te in folds:
        tr = np.setdiff1d(np.arange(len(yd)), te)
        c0, c1 = Xd[tr][yd[tr] == 0].mean(0), Xd[tr][yd[tr] == 1].mean(0)
        right += np.sum((((Xd[te] - c1) ** 2).sum(1) < ((Xd[te] - c0) ** 2).sum(1)) == yd[te])
    return right / len(yd)

gB = np.random.default_rng(5962)
cv_scores = []
for d in range(200):
    yd = np.tile([0, 1], 50)
    Xd = 0.3 * yd[:, None] * np.ones(10) + gB.normal(size=(100, 10))
    cv_scores.append(...)
a = np.mean(cv_scores)
binom_se = np.sqrt(a * (1 - a) / 100)
lo, hi = np.percentile(cv_scores, [2.5, 97.5])
print(f"mean CV accuracy {a:.3f}; spread across datasets {np.std(cv_scores):.3f}; binomial standard error {binom_se:.3f}")
print(f"95% of datasets report between {lo:.2f} and {hi:.2f}")

In [ ]:
import numpy as np
def test_spread():
    "across datasets, cross-validated accuracies spread at least as much as the binomial standard error"
    assert len(cv_scores) == 200 and np.std(cv_scores) > 0.9 * binom_se
def test_range():
    "the middle 95% of estimates spans more than 15 percentage points"
    assert np.percentile(cv_scores, 97.5) - np.percentile(cv_scores, 2.5) > 0.15

test_spread()
test_range()
print("All checks passed")